In [10]:
import json
from collections import defaultdict
from decimal import Decimal
import pandas as pd
from web3 import Web3
import os
from dotenv import load_dotenv

load_dotenv()

True

In [11]:
w3 = Web3(Web3.HTTPProvider("https://api.avax.network/ext/bc/C/rpc"))
JOETROLLER = "0xdc13687554205E5b89Ac783db14bb5bba4A1eDaC"
LIQUIDATION_PREMIUM = Decimal("1.015")
DUST = Decimal("1")  # $1
BLOCK = os.getenv("BLOCK")
if BLOCK is None:
    BLOCK = "latest"
else:
    BLOCK = int(BLOCK)
print(f"Using block {BLOCK}")

Using block 84339406


In [12]:
def parse_user(field, value):
    if field == "address":
        return value
    return None


def parse_position(field, value, xjoe_to_joe):
    if field.endswith("(supply)"):
        asset = field.replace(" (supply)", "")
        amount = Decimal(value)
    elif field.endswith("(borrow)"):
        asset = field.replace(" (borrow)", "")
        amount = -Decimal(value)
    else:
        return None
    # Treat xJOE positions as raw JOE so the classifier sees a single JOE
    # market. Both tokens are 18-decimal; the ratio is xJOE -> JOE.
    if asset == "jXJOE":
        asset = "jJOE"
        amount *= xjoe_to_joe
    return (asset, amount)


def load_joetroller(w3):
    with open("out/Joetroller.sol/Joetroller.json") as f:
        joetroller_abi = json.load(f)["abi"]
    return w3.eth.contract(address=JOETROLLER, abi=joetroller_abi)


def load_oracle(w3, joetroller):
    with open("out/PriceOracle.sol/PriceOracle.json") as f:
        oracle_abi = json.load(f)["abi"]
    return w3.eth.contract(address=joetroller.functions.oracle().call(block_identifier=BLOCK), abi=oracle_abi)


def load_markets(w3, joetroller, oracle):
    with open("out/JCollateralCapErc20.sol/JCollateralCapErc20.json") as f:
        jtoken_abi = json.load(f)["abi"]
    markets = {}
    for market in joetroller.functions.getAllMarkets().call(block_identifier=BLOCK):
        m = w3.eth.contract(address=market, abi=jtoken_abi)
        symbol = m.functions.symbol().call(block_identifier=BLOCK)
        underlying = w3.eth.contract(
            address=m.functions.underlying().call(block_identifier=BLOCK), abi=jtoken_abi
        )

        decimals = underlying.functions.decimals().call(block_identifier=BLOCK)
        markets[symbol] = {
            "address": market,
            "underlying": underlying.address,
            "decimals": decimals,
            "collateralFactor": Decimal(joetroller.functions.markets(market).call(block_identifier=BLOCK)[1])
            / 10**18,
            "price": Decimal(oracle.functions.getUnderlyingPrice(market).call(block_identifier=BLOCK))
            / 10 ** (36 - decimals),
            "totalBorrows": Decimal(m.functions.totalBorrowsCurrent().call(block_identifier=BLOCK))
            / 10**decimals,
            "totalReserves": Decimal(m.functions.totalReserves().call(block_identifier=BLOCK)) / 10**decimals,
            "balance": Decimal(underlying.functions.balanceOf(market).call(block_identifier=BLOCK))
            / 10**decimals,
        }
    return markets


def load_xjoe_to_joe_ratio(w3, markets):
    # Same formula as PriceOracleProxyUSD.getXJoeRatio:
    #   ratio = JOE.balanceOf(xJOE) / xJOE.totalSupply()
    erc20_abi = [
        {"constant": True, "inputs": [{"name": "", "type": "address"}], "name": "balanceOf",
         "outputs": [{"name": "", "type": "uint256"}], "type": "function"},
        {"constant": True, "inputs": [], "name": "totalSupply",
         "outputs": [{"name": "", "type": "uint256"}], "type": "function"},
    ]
    xjoe_address = markets["jXJOE"]["underlying"]
    joe = w3.eth.contract(address=markets["jJOE"]["underlying"], abi=erc20_abi)
    xjoe = w3.eth.contract(address=xjoe_address, abi=erc20_abi)
    joe_in_xjoe = Decimal(joe.functions.balanceOf(xjoe_address).call(block_identifier=BLOCK))
    xjoe_supply = Decimal(xjoe.functions.totalSupply().call(block_identifier=BLOCK))
    return joe_in_xjoe / xjoe_supply


def merge_xjoe_into_joe(markets, xjoe_to_joe):
    # Fold xJOE-denominated state into jJOE (JOE-equivalent), then drop jXJOE
    # so downstream code sees a single JOE market.
    xjoe = markets["jXJOE"]
    joe = markets["jJOE"]
    for key in ("balance", "totalBorrows", "totalReserves"):
        joe[key] += xjoe[key] * xjoe_to_joe
    del markets["jXJOE"]


def load_all_positions(prices, xjoe_to_joe):
    with open("all-user-positions.csv") as f:
        fields = f.readline().strip().split(",")
        all_positions = {}
        total_borrows = defaultdict(Decimal)
        for line in f:
            values = line.strip().split(",")
            user = parse_user(fields[0], values[0])
            positions = defaultdict(Decimal)
            for i in range(1, len(fields)):
                field = fields[i]
                position = parse_position(field, values[i], xjoe_to_joe)
                if position is None:
                    continue
                asset, amount = position
                # For users that have both deposited and borrowed the same asset,
                # we just compute the net position without any fee.
                positions[asset] += amount * prices[asset]
                if amount < 0:
                    total_borrows[asset] += abs(amount)
            all_positions[user] = positions
    all_positions = pd.DataFrame(all_positions)
    return all_positions, total_borrows

In [13]:
joetroller = load_joetroller(w3)
oracle = load_oracle(w3, joetroller)
markets = load_markets(w3, joetroller, oracle)
xjoe_to_joe = load_xjoe_to_joe_ratio(w3, markets)
merge_xjoe_into_joe(markets, xjoe_to_joe)
print(f"xJOE -> JOE ratio: {xjoe_to_joe:.6f}")

xJOE -> JOE ratio: 1.318229


In [14]:
prices = pd.Series({asset: markets[asset]["price"] for asset in markets})
reserves = pd.Series(
    {asset: markets[asset]["totalReserves"] * prices[asset] for asset in markets}
)
balances_raw = pd.Series({asset: markets[asset]["balance"] for asset in markets})

(all_positions, total_borrows) = load_all_positions(prices, xjoe_to_joe)

In [15]:
def liquidate(positions, bad_debt):
    while True:
        borrow_asset = positions.idxmin()
        borrow_amount = positions[borrow_asset]
        if borrow_amount >= 0:  # No more borrows
            for asset, amount in positions.items():
                # Only redeem amount greater than DUST
                if amount < DUST:
                    positions[asset] = 0
            return positions
        deposit_asset = positions.idxmax()
        deposit_amount = positions[deposit_asset]
        if deposit_amount <= 0:  # No more deposits, must be underwater
            bad_debt[borrow_asset] += abs(borrow_amount)
            positions[borrow_asset] = Decimal(0)
            continue

        liq_amount = abs(borrow_amount) * LIQUIDATION_PREMIUM
        if liq_amount <= deposit_amount:
            positions[deposit_asset] -= liq_amount
            positions[borrow_asset] = Decimal(0)
        else:
            positions[deposit_asset] = Decimal(0)
            # Borrows are stored as negative, so we add the deposit amount to reduce it
            positions[borrow_asset] += deposit_amount / LIQUIDATION_PREMIUM


all_deposits = {}
bad_debt = defaultdict(Decimal)

for user, positions in all_positions.items():
    redeems = liquidate(positions, bad_debt)
    all_deposits[user] = redeems
all_deposits = pd.DataFrame(all_deposits)
bad_debt = pd.Series(bad_debt)

phantom_debt = pd.Series(
    {
        asset: (markets[asset]["totalBorrows"] - total_borrows[asset]) * prices[asset]
        for asset in markets
    }
)

shortfall = reserves - (bad_debt + phantom_debt)
total_debt = shortfall.sum()

# This should never happen
assert total_debt < 0

In [16]:
total_deposits = all_deposits.sum().sum()

redemption_fee = abs(total_debt) / total_deposits
redemption_fee

all_redeems = all_deposits * (Decimal(1) - redemption_fee)

escrow = {
    user: {asset: int(amt * 10**markets[asset]["decimals"] / prices[asset]) for asset, amt in col.items() if amt != 0}
    for user, col in all_redeems.items()
    if (col != 0).any()
}

with open("escrow.json", "w") as f:
    json.dump(escrow, f)

In [17]:
print(f"Liquidation fee: {(LIQUIDATION_PREMIUM - 1) * 100:.2f}%")
print(f"Redemption fee:  {redemption_fee * 100:.2f}%")

print(f"\nReserves:        ${float(reserves.sum()):,.2f}")
print(f"Bad debt:        ${-float(bad_debt.sum()):,.2f}")
print(f"Phantom debt:    ${-float(phantom_debt.sum()):,.2f}")
print(f"Total debt:      ${float(total_debt):,.2f}")

print(
    f"\n{'Asset':<14} {'Required ($)':>14} {'Available ($)':>14} {'Delta ($)':>14} {'Delta':>18}"
)
print("-" * 78)
(sum_req_usd, sum_avail_usd, delta_total_usd) = (Decimal(0), Decimal(0), Decimal(0))
for asset in markets:
    required_usd = all_redeems.loc[asset].sum()
    available_usd = balances_raw[asset] * prices[asset]
    delta_usd = available_usd - required_usd
    delta_raw = delta_usd / prices[asset]
    sum_req_usd += required_usd
    sum_avail_usd += available_usd
    delta_total_usd += delta_usd
    print(
        f"{asset:<14} {float(required_usd):>14,.0f} {float(available_usd):>14,.0f}"
        f" {float(delta_usd):>14,.0f} {float(delta_raw):>18,.4f}"
    )
print("-" * 78)
print(
    f"{'TOTAL':<14} {float(sum_req_usd):>14,.0f} {float(sum_avail_usd):>14,.0f}"
    f" {float(delta_total_usd):>14,.0f}"
)

Liquidation fee: 1.50%
Redemption fee:  4.06%

Reserves:        $30,366.92
Bad debt:        $-6,526.00
Phantom debt:    $-136,829.65
Total debt:      $-113,055.21

Asset            Required ($)  Available ($)      Delta ($)              Delta
------------------------------------------------------------------------------
jAVAX                 201,264        216,251         14,987         1,640.6825
jWETH                 445,570        539,141         93,571            40.9164
jWBTC                 311,703        382,274         70,571             0.9105
jUSDC                 619,018        458,946       -160,071      -160,108.1403
jUSDT                 876,981        766,616       -110,365      -110,409.6483
jDAI                   23,199         11,841        -11,359       -11,360.4954
jLINK                  48,249         70,523         22,274         2,435.4314
jUSDTNative            45,241         89,624         44,383        44,400.7836
jUSDCNative            36,645         31,070  